# 基金週報（追蹤 + 投組回測）Notebook（PDF：思源黑體）

本 Notebook 會：

- 讀取 `fund_list.xlsx`（基金代號 / 基金名稱 / 查詢傳輸代號 / 權重）
- 自動判斷國內/海外 BCD API（先試 `tBCDNavList`，失敗再試 `BCDNavList`）
- 產出基金週報：上週、近4週、近12週、回撤與回撤期間
- 告警：上週<-1%、近4週<-2%、回撤<-5%（基金 + 投組；投組含再平衡/不再平衡）
- 投組回測：等權/自訂權重 × 再平衡/不再平衡
- 風險貢獻（Risk Contribution）
- 產圖 PNG 並貼到 Excel（charts_p1/charts_p2，每頁2張大圖）
- 產出 PDF（Summary 文字頁 + 兩頁圖表），中文字型使用 **思源黑體**：`D:\fonts\NotoSansTC-Regular.ttf`

> 若找不到該字型檔，PDF 會自動 fallback 到 ReportLab 內建 `MSung-Light`。


In [ ]:
# 第一次執行需要安裝（有裝可略過）
# !pip install requests pandas openpyxl matplotlib reportlab


In [ ]:
import re
import os
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datetime import date, timedelta, datetime
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage
from openpyxl.styles import Font


## 1) 參數設定
建議每週一執行，週期以「上週五收盤」為結束日（W-FRI）。


In [ ]:
INPUT_EXCEL  = "fund_list.xlsx"
OUTPUT_EXCEL = "weekly_fund_report.xlsx"
OUTPUT_PDF   = "weekly_fund_report.pdf"

DAYS_BACK = 365
REBALANCE = "M"          # 'M' 每月、'Q' 每季、None 不再平衡
RISK_FREE = 0.0
TIMEOUT = 30

# 告警門檻（三條件同時成立）
TH_WEEK = -0.01
TH_4W   = -0.02
TH_DD   = -0.05

# 嚴重度分數權重
SEV_W_DD   = 0.5
SEV_W_4W   = 0.3
SEV_W_WEEK = 0.2

CHART_DIR = "charts"
os.makedirs(CHART_DIR, exist_ok=True)


## 2) BCD 下載與解析（自動判斷國內/海外）


In [ ]:
def fmt_ymd(d: date) -> str:
    return f"{d.year}-{d.month}-{d.day}"


def fetch_bcd_raw(endpoint: str, a: str, start: str, end: str, b: int = 1, timeout: int = 30) -> str:
    url = f"https://fund.bot.com.tw/w/bcd/{endpoint}.djbcd"
    params = {"a": a, "b": b, "c": start, "d": end}
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, params=params, headers=headers, timeout=timeout)
    r.raise_for_status()
    return r.text.strip()


def parse_bcd_nav(raw: str) -> pd.DataFrame:
    parts = re.split(r"\s+", raw.strip(), maxsplit=1)
    if len(parts) < 2:
        raise ValueError("BCD 回傳無法分成兩段")

    dates = [x for x in parts[0].split(",") if x]
    navs  = [x for x in parts[1].split(",") if x]

    if not dates or not navs:
        raise ValueError("日期或淨值序列為空")
    if len(dates) != len(navs):
        raise ValueError(f"日期/淨值筆數不一致：{len(dates)} vs {len(navs)}")

    df = pd.DataFrame({
        "日期": pd.to_datetime(dates, format="%Y%m%d", errors="coerce"),
        "淨值": pd.to_numeric(navs, errors="coerce")
    }).dropna()

    if df.empty:
        raise ValueError("解析後 DataFrame 為空")

    return df.sort_values("日期").reset_index(drop=True)


def detect_fund_type_and_fetch(a: str, start: str, end: str, timeout: int = 30) -> tuple[str, pd.DataFrame]:
    # 先試國內
    try:
        raw = fetch_bcd_raw("tBCDNavList", a=a, start=start, end=end, timeout=timeout)
        return "國內", parse_bcd_nav(raw)
    except Exception:
        pass

    # 再試海外
    raw = fetch_bcd_raw("BCDNavList", a=a, start=start, end=end, timeout=timeout)
    return "海外", parse_bcd_nav(raw)


def add_change_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values("日期").copy()
    df["漲/跌"] = df["淨值"].diff()
    df["漲跌幅(%)"] = (df["漲/跌"] / df["淨值"].shift(1)) * 100
    return df


## 3) 讀取基金清單（fund_list.xlsx）


In [ ]:
fund_list = pd.read_excel(INPUT_EXCEL, engine="openpyxl", dtype=str).fillna("")

required = ["基金代號", "基金名稱", "查詢傳輸代號"]
missing = [c for c in required if c not in fund_list.columns]
if missing:
    raise ValueError(f"Excel 缺少欄位：{missing}")

for c in required:
    fund_list[c] = fund_list[c].astype(str).str.strip()

if "權重" not in fund_list.columns:
    fund_list["權重"] = ""
else:
    fund_list["權重"] = fund_list["權重"].astype(str).str.strip()

fund_list = fund_list[(fund_list["基金代號"] != "") & (fund_list["查詢傳輸代號"] != "")].copy()
fund_list.head()


## 4) 批次抓取淨值（長表）


In [ ]:
end_date = date.today()
start_date = end_date - timedelta(days=DAYS_BACK)
start = fmt_ymd(start_date)
end   = fmt_ymd(end_date)
print("抓取區間：", start, "~", end)

all_rows, fail_rows = [], []

for _, r in fund_list.iterrows():
    code_ = r["基金代號"]
    name_ = r["基金名稱"]
    a_    = r["查詢傳輸代號"]

    try:
        ftype, nav = detect_fund_type_and_fetch(a=a_, start=start, end=end, timeout=TIMEOUT)
        nav = add_change_cols(nav)
        nav.insert(0, "查詢傳輸代號", a_)
        nav.insert(0, "基金名稱", name_)
        nav.insert(0, "基金代號", code_)
        nav.insert(3, "基金類型", ftype)
        all_rows.append(nav)
        print(f"[OK] {code_} | {a_} | {ftype} | rows={len(nav)}")
    except Exception as e:
        fail_rows.append({"基金代號": code_, "基金名稱": name_, "查詢傳輸代號": a_, "錯誤": str(e)})
        print(f"[FAIL] {code_} | {a_} -> {e}")

result_df = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
fail_df = pd.DataFrame(fail_rows)
result_df.head()


## 5) 建立日報酬/淨值矩陣


In [ ]:
def build_matrices(df_long: pd.DataFrame):
    d = df_long.sort_values("日期").copy()
    d["日報酬"] = d.groupby("基金代號")["淨值"].pct_change()
    ret = d.pivot_table(index="日期", columns="基金代號", values="日報酬")
    nav = d.pivot_table(index="日期", columns="基金代號", values="淨值")
    return ret, nav

ret_mat, nav_mat = build_matrices(result_df)
common_dates = ret_mat.dropna(how="all").index
ret_mat = ret_mat.loc[common_dates]
nav_mat = nav_mat.loc[common_dates]

name_map = (result_df[["基金代號", "基金名稱"]]
            .drop_duplicates()
            .set_index("基金代號")["基金名稱"].to_dict())

ret_mat.tail()


## 6) 權重：等權 / 自訂權重


In [ ]:
def get_equal_weights(codes):
    codes = list(codes)
    return {c: 1/len(codes) for c in codes}


def get_custom_weights(fund_list_df: pd.DataFrame, codes_in_data):
    tmp = fund_list_df[fund_list_df["基金代號"].isin(codes_in_data)].copy()
    tmp["權重_num"] = pd.to_numeric(tmp["權重"].replace("", np.nan), errors="coerce")
    tmp = tmp.dropna(subset=["權重_num"])
    if tmp.empty:
        return None, "自訂權重：未提供（將只輸出等權投組）"
    w = dict(zip(tmp["基金代號"], tmp["權重_num"]))
    s = sum(w.values())
    if s <= 0:
        return None, "自訂權重：總和<=0（將只輸出等權投組）"
    w = {k: v/s for k, v in w.items()}
    return w, "自訂權重：已使用 Excel 權重並正規化"

codes = ret_mat.columns.tolist()
w_equal = get_equal_weights(codes)
w_custom, w_msg = get_custom_weights(fund_list, codes)
print(w_msg)


## 7) 投組回測（等權/自訂 × 再平衡/不再平衡）


In [ ]:
def portfolio_backtest(ret_matrix: pd.DataFrame, weights: dict, rebalance="M"):
    r = ret_matrix.copy()
    cols = [c for c in r.columns if c in weights]
    r = r[cols].dropna(how="all")

    w0 = pd.Series({c: float(weights[c]) for c in cols})
    w0 = w0 / w0.sum()

    if rebalance is None:
        port_ret = (r.fillna(0) * w0).sum(axis=1)
        port_nav = (1 + port_ret).cumprod()
        return pd.DataFrame({"投組日報酬": port_ret, "投組淨值": port_nav})

    port_ret = []
    current_w = w0.copy()
    prev_period = None

    for dt, row in r.iterrows():
        period = dt.to_period(rebalance)
        if prev_period is None or period != prev_period:
            current_w = w0.copy()
        prev_period = period

        row0 = row.fillna(0)
        pr = (row0 * current_w).sum()
        port_ret.append(pr)

        grow = (1 + row0)
        current_w = (current_w * grow)
        current_w = current_w / current_w.sum() if current_w.sum() != 0 else w0.copy()

    port_ret = pd.Series(port_ret, index=r.index, name="投組日報酬")
    port_nav = (1 + port_ret).cumprod()
    return pd.DataFrame({"投組日報酬": port_ret, "投組淨值": port_nav})


def perf_metrics(port_df: pd.DataFrame, rf=0.0, freq=252):
    s = port_df["投組日報酬"].dropna()
    if s.empty:
        return {}
    total = (1 + s).prod() - 1
    vol = s.std() * np.sqrt(freq)
    sharpe = (total - rf) / vol if vol > 0 else np.nan
    equity = (1 + s).cumprod()
    dd = equity / equity.cummax() - 1
    mdd = dd.min()
    days = (s.index[-1] - s.index[0]).days
    cagr = (1 + total) ** (365/days) - 1 if days > 0 else np.nan
    return {"累積報酬": total, "CAGR": cagr, "年化波動": vol, "Sharpe(簡化)": sharpe, "最大回撤": mdd}

port_equal_rb = portfolio_backtest(ret_mat, w_equal, rebalance=REBALANCE)
port_equal_bh = portfolio_backtest(ret_mat, w_equal, rebalance=None)

port_custom_rb = None
port_custom_bh = None
if w_custom is not None:
    port_custom_rb = portfolio_backtest(ret_mat, w_custom, rebalance=REBALANCE)
    port_custom_bh = portfolio_backtest(ret_mat, w_custom, rebalance=None)

portfolios = {
    f"等權投組(再平衡={REBALANCE})": port_equal_rb,
    "等權投組(不再平衡)": port_equal_bh,
}
if port_custom_rb is not None:
    portfolios[f"自訂權重投組(再平衡={REBALANCE})"] = port_custom_rb
if port_custom_bh is not None:
    portfolios["自訂權重投組(不再平衡)"] = port_custom_bh

portfolio_compare_all = pd.DataFrame([
    {"投組": name, **perf_metrics(pdf, rf=RISK_FREE)}
    for name, pdf in portfolios.items()
]).sort_values("CAGR", ascending=False).reset_index(drop=True)

portfolio_compare_all


## 8) 週報計算（W-FRI） + 回撤 + 告警


In [ ]:
def weekly_nav_and_ret(nav_series: pd.Series):
    wnav = nav_series.resample("W-FRI").last().dropna()
    wret = wnav.pct_change()
    return wnav, wret


def weekly_snapshot(weekly_ret: pd.Series):
    weekly_ret = weekly_ret.dropna()
    if len(weekly_ret) < 2:
        return {"上週報酬": np.nan, "近4週報酬": np.nan, "近12週報酬": np.nan, "週末日": pd.NaT}
    last = weekly_ret.iloc[-1]
    w4 = (1 + weekly_ret.tail(4)).prod() - 1 if len(weekly_ret) >= 4 else np.nan
    w12 = (1 + weekly_ret.tail(12)).prod() - 1 if len(weekly_ret) >= 12 else np.nan
    return {"上週報酬": float(last), "近4週報酬": float(w4), "近12週報酬": float(w12), "週末日": weekly_ret.index[-1].date()}


def drawdown_details(nav_series: pd.Series) -> dict:
    s = nav_series.dropna().copy()
    if s.empty:
        return {"目前回撤": np.nan, "高點日期": pd.NaT, "回撤開始日": pd.NaT, "距離高點天數": np.nan, "回撤持續天數": np.nan}

    peak = s.cummax()
    dd = s / peak - 1
    cur_dd = float(dd.iloc[-1])

    is_peak = np.isclose(dd.values, 0.0, atol=1e-12)
    last_peak_idx = np.where(is_peak)[0][-1] if is_peak.any() else 0
    peak_date = s.index[last_peak_idx]

    after_peak = s.loc[peak_date:]
    after_peak_peak = after_peak.cummax()
    below = (after_peak < after_peak_peak)

    if below.any():
        dd_start_date = below[below].index[0]
        dd_duration = int((s.index.get_loc(s.index[-1]) - s.index.get_loc(dd_start_date)))
        dd_start_out = dd_start_date.date()
    else:
        dd_start_out = pd.NaT
        dd_duration = 0

    days_since_peak = int((s.index.get_loc(s.index[-1]) - s.index.get_loc(peak_date)))

    return {
        "目前回撤": cur_dd,
        "高點日期": peak_date.date(),
        "回撤開始日": dd_start_out,
        "距離高點天數": days_since_peak,
        "回撤持續天數": dd_duration
    }

# 基金週報
fund_rows = []
for code_ in nav_mat.columns:
    s = nav_mat[code_].dropna()
    if s.empty:
        continue
    _, wret = weekly_nav_and_ret(s)
    snap = weekly_snapshot(wret)
    dd = drawdown_details(s)
    fund_rows.append({"基金代號": code_, "基金名稱": name_map.get(code_, ""), **snap, **dd})

fund_weekly2 = pd.DataFrame(fund_rows)

# 投組週報
port_weekly_rows = []
port_dd_rows = []
for name, pdf in portfolios.items():
    nav = pdf["投組淨值"]
    _, wret = weekly_nav_and_ret(nav)
    snap = weekly_snapshot(wret)
    dd = drawdown_details(nav)
    port_weekly_rows.append({"投組": name, **snap})
    port_dd_rows.append({"投組": name, **dd})

portfolio_weekly_all = pd.DataFrame(port_weekly_rows)
port_dd = pd.DataFrame(port_dd_rows)

# alerts：基金 + 投組
alerts_fund = fund_weekly2[(fund_weekly2["上週報酬"] < TH_WEEK) &
                           (fund_weekly2["近4週報酬"] < TH_4W) &
                           (fund_weekly2["目前回撤"] < TH_DD)].copy()
alerts_fund["類別"] = "基金"
alerts_fund["標的"] = alerts_fund["基金代號"] + " " + alerts_fund["基金名稱"]

alerts_port = portfolio_weekly_all.merge(port_dd, on="投組", how="left")
alerts_port = alerts_port[(alerts_port["上週報酬"] < TH_WEEK) &
                          (alerts_port["近4週報酬"] < TH_4W) &
                          (alerts_port["目前回撤"] < TH_DD)].copy()
alerts_port["類別"] = "投組"
alerts_port["標的"] = alerts_port["投組"]

alerts = pd.concat([
    alerts_fund[["類別","標的","週末日","上週報酬","近4週報酬","近12週報酬","目前回撤","高點日期","回撤開始日","距離高點天數","回撤持續天數"]],
    alerts_port[["類別","標的","週末日","上週報酬","近4週報酬","近12週報酬","目前回撤","高點日期","回撤開始日","距離高點天數","回撤持續天數"]],
], ignore_index=True)

alerts["嚴重度分數"] = (
    SEV_W_DD   * (-alerts["目前回撤"]).clip(lower=0) +
    SEV_W_4W   * (-alerts["近4週報酬"]).clip(lower=0) +
    SEV_W_WEEK * (-alerts["上週報酬"]).clip(lower=0)
)
alerts = alerts.sort_values(["嚴重度分數","目前回撤"], ascending=[False, True]).reset_index(drop=True)

week_end = pd.to_datetime(portfolio_weekly_all["週末日"].dropna().iloc[0]) if portfolio_weekly_all["週末日"].notna().any() else pd.NaT

fund_weekly2.head(), portfolio_weekly_all.head(), alerts.head()


## 9) 相關矩陣 + 風險貢獻


In [ ]:
corr = ret_mat.corr(min_periods=60)

def risk_contribution(ret_matrix: pd.DataFrame, weights: dict, freq=252):
    r = ret_matrix.copy()
    cols = [c for c in r.columns if c in weights]
    r = r[cols].dropna(how="all")
    if r.empty:
        return pd.DataFrame(), np.nan

    w = pd.Series({c: float(weights[c]) for c in cols})
    w = w / w.sum()

    cov = r.dropna().cov()
    if cov.empty:
        return pd.DataFrame(), np.nan

    cov_ann = cov * freq
    port_var = float(w.values @ cov_ann.values @ w.values)
    port_vol = np.sqrt(port_var) if port_var > 0 else np.nan

    mrc = cov_ann.values @ w.values
    rc_num = w.values * mrc
    rc_pct = rc_num / port_var if port_var > 0 else np.nan
    rc_abs = rc_pct * port_vol if port_vol == port_vol else np.nan

    out = pd.DataFrame({
        "基金代號": cols,
        "基金名稱": [name_map.get(c, "") for c in cols],
        "權重": w.values,
        "RC%": rc_pct,
        "AbsRC(年化波動貢獻)": rc_abs
    }).sort_values("RC%", ascending=False).reset_index(drop=True)

    return out, port_vol

rc_equal, vol_equal = risk_contribution(ret_mat, w_equal)
rc_custom, vol_custom = (pd.DataFrame(), np.nan)
if w_custom is not None:
    rc_custom, vol_custom = risk_contribution(ret_mat, w_custom)

corr.iloc[:5,:5], rc_equal.head()


## 10) 產生圖表 PNG（每頁 2 張放大）


In [ ]:
def save_portfolio_nav_2panels(port_equal_rb, port_equal_bh, port_custom_rb=None, port_custom_bh=None, path=None):
    fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)

    axes[0].plot(port_equal_rb.index, port_equal_rb["投組淨值"], label="等權(再平衡)", linewidth=2)
    if port_custom_rb is not None:
        axes[0].plot(port_custom_rb.index, port_custom_rb["投組淨值"], label="自訂權重(再平衡)", linewidth=2)
    axes[0].set_title("投組淨值走勢（上：再平衡）")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(port_equal_bh.index, port_equal_bh["投組淨值"], label="等權(不再平衡)", linewidth=2)
    if port_custom_bh is not None:
        axes[1].plot(port_custom_bh.index, port_custom_bh["投組淨值"], label="自訂權重(不再平衡)", linewidth=2)
    axes[1].set_title("投組淨值走勢（下：不再平衡 / Buy & Hold）")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    if path:
        plt.savefig(path, dpi=220)
    plt.close()


def save_alerts_bar(alerts_df: pd.DataFrame, path=None):
    plt.figure(figsize=(12, 5))
    if alerts_df.empty:
        plt.text(0.5, 0.5, "本週無符合三條件告警的基金/投組 ✅", ha="center", va="center", fontsize=16)
        plt.axis("off")
    else:
        x = alerts_df["標的"]
        y = alerts_df["目前回撤"]
        plt.bar(x, y)
        plt.title("告警清單：目前回撤（越低越嚴重）")
        plt.xlabel("標的（基金/投組）")
        plt.ylabel("目前回撤")
        plt.xticks(rotation=30, ha="right")
        plt.grid(True, axis="y", alpha=0.3)
    if path:
        plt.tight_layout()
        plt.savefig(path, dpi=220)
    plt.close()


def save_risk_return_scatter(ret_mat: pd.DataFrame, path=None, freq=252):
    ann_ret = ret_mat.mean() * freq
    ann_vol = ret_mat.std() * (freq ** 0.5)
    plt.figure(figsize=(12, 6))
    plt.scatter(ann_vol, ann_ret, alpha=0.8)
    for code_ in ret_mat.columns:
        plt.text(ann_vol[code_], ann_ret[code_], code_, fontsize=9, alpha=0.85)
    plt.title("風險報酬散點圖（年化報酬 vs 年化波動）")
    plt.xlabel("年化波動（Vol）")
    plt.ylabel("年化報酬（簡化估算）")
    plt.grid(True, alpha=0.3)
    if path:
        plt.tight_layout()
        plt.savefig(path, dpi=220)
    plt.close()


def save_corr_heatmap(corr_df: pd.DataFrame, path=None):
    plt.figure(figsize=(12, 6))
    plt.imshow(corr_df.values, aspect="auto", cmap="coolwarm", vmin=-1, vmax=1)
    plt.colorbar(label="Correlation")
    plt.title("基金日報酬相關係數熱圖")
    plt.xticks(range(len(corr_df.columns)), corr_df.columns, rotation=90, fontsize=7)
    plt.yticks(range(len(corr_df.index)), corr_df.index, fontsize=7)
    plt.tight_layout()
    if path:
        plt.savefig(path, dpi=220)
    plt.close()

png_port   = os.path.join(CHART_DIR, "p1_01_portfolio_nav.png")
png_alerts = os.path.join(CHART_DIR, "p1_02_alerts.png")
png_scatter= os.path.join(CHART_DIR, "p2_01_risk_return.png")
png_corr   = os.path.join(CHART_DIR, "p2_02_corr_heatmap.png")

save_portfolio_nav_2panels(port_equal_rb, port_equal_bh, port_custom_rb, port_custom_bh, png_port)
save_alerts_bar(alerts, png_alerts)
save_risk_return_scatter(ret_mat.dropna(how="all"), png_scatter)
save_corr_heatmap(corr, png_corr)

[png_port, png_alerts, png_scatter, png_corr]


## 11) Summary（首頁資料）


In [ ]:
run_time = datetime.now()

summary_kpis = pd.DataFrame([
    ["報表產出時間", run_time.strftime("%Y-%m-%d %H:%M")],
    ["週報週期結束日(週五)", (week_end.strftime("%Y-%m-%d") if str(week_end) != "NaT" else "")],
    ["基金數量", int(fund_list.shape[0])],
    ["告警數量-基金", int((alerts["類別"]=="基金").sum())],
    ["告警數量-投組", int((alerts["類別"]=="投組").sum())],
], columns=["項目", "值"])

top_rc_equal = rc_equal.head(5).copy() if not rc_equal.empty else pd.DataFrame()
top_rc_custom = rc_custom.head(5).copy() if (w_custom is not None and not rc_custom.empty) else pd.DataFrame()

summary_kpis


## 12) 匯出 Excel（含圖表嵌入）


In [ ]:
with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
    # Summary 區塊
    summary_kpis.to_excel(writer, index=False, sheet_name="summary", startrow=0)
    portfolio_weekly_all.to_excel(writer, index=False, sheet_name="summary", startrow=10)
    portfolio_compare_all.to_excel(writer, index=False, sheet_name="summary", startrow=18)

    start_row = 28
    if not top_rc_equal.empty:
        top_rc_equal.to_excel(writer, index=False, sheet_name="summary", startrow=start_row)
        start_row += 8
    if not top_rc_custom.empty:
        top_rc_custom.to_excel(writer, index=False, sheet_name="summary", startrow=start_row)

    # 明細
    result_df.to_excel(writer, index=False, sheet_name="nav_long")
    fund_weekly2.to_excel(writer, index=False, sheet_name="fund_weekly")
    alerts.to_excel(writer, index=False, sheet_name="alerts")

    port_equal_rb.to_excel(writer, index=True, sheet_name="port_equal_rb_daily")
    port_equal_bh.to_excel(writer, index=True, sheet_name="port_equal_bh_daily")
    if port_custom_rb is not None:
        port_custom_rb.to_excel(writer, index=True, sheet_name="port_custom_rb_daily")
    if port_custom_bh is not None:
        port_custom_bh.to_excel(writer, index=True, sheet_name="port_custom_bh_daily")

    corr.to_excel(writer, sheet_name="corr")
    fund_dd = fund_weekly2[["基金代號","基金名稱","目前回撤","高點日期","回撤開始日","距離高點天數","回撤持續天數"]].copy()
    fund_dd.to_excel(writer, index=False, sheet_name="drawdown_fund")
    port_dd.to_excel(writer, index=False, sheet_name="drawdown_port")

    rc_equal.to_excel(writer, index=False, sheet_name="risk_contrib_equal")
    if w_custom is not None and not rc_custom.empty:
        rc_custom.to_excel(writer, index=False, sheet_name="risk_contrib_custom")

    if not fail_df.empty:
        fail_df.to_excel(writer, index=False, sheet_name="fail")

# 插圖（charts_p1 / charts_p2）
wb = load_workbook(OUTPUT_EXCEL)
for s in ["charts_p1", "charts_p2"]:
    if s in wb.sheetnames:
        wb.remove(wb[s])
ws1 = wb.create_sheet("charts_p1", 0)
ws2 = wb.create_sheet("charts_p2", 1)
ws1["A1"] = "Weekly Report - Page 1（投組＆告警）"
ws2["A1"] = "Weekly Report - Page 2（風險＆關聯）"
ws1["A1"].font = Font(bold=True, size=16)
ws2["A1"].font = Font(bold=True, size=16)

def add_img(ws, file_path, anchor):
    if os.path.exists(file_path):
        ws.add_image(XLImage(file_path), anchor)

add_img(ws1, png_port,   "A3")
add_img(ws1, png_alerts, "A28")
add_img(ws2, png_scatter,"A3")
add_img(ws2, png_corr,   "A28")

# alerts 嚴重度分數顯示為百分比
if "alerts" in wb.sheetnames:
    ws = wb["alerts"]
    col_idx = None
    for cell in ws[1]:
        if cell.value == "嚴重度分數":
            col_idx = cell.col_idx
            break
    if col_idx:
        for r in range(2, ws.max_row+1):
            c = ws.cell(r, col_idx)
            if isinstance(c.value, (int, float)):
                c.number_format = "0.00%"

wb.save(OUTPUT_EXCEL)
OUTPUT_EXCEL


## 13) 產出 PDF（字型：思源黑體 NotoSansTC）


In [ ]:
from reportlab.lib.pagesizes import A4, landscape
from reportlab.pdfgen import canvas
from reportlab.lib.utils import ImageReader
from reportlab.lib.units import cm
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.pdfbase.cidfonts import UnicodeCIDFont

FONT_PATH = r"D:\fonts\NotoSansTC-Regular.ttf"
CH_FONT = "NotoSansTC"

# 註冊思源黑體（若找不到字型檔則 fallback）
if os.path.exists(FONT_PATH):
    pdfmetrics.registerFont(TTFont(CH_FONT, FONT_PATH))
else:
    CH_FONT = "MSung-Light"
    pdfmetrics.registerFont(UnicodeCIDFont(CH_FONT))
    print(f"[WARN] 找不到字型檔：{FONT_PATH}，已改用內建字型 {CH_FONT}")


def draw_title(c, title, subtitle=""):
    c.setFont("Helvetica-Bold", 18)
    c.drawString(1.2*cm, 19.8*cm, title)
    c.setFont("Helvetica", 11)
    if subtitle:
        c.drawString(1.2*cm, 19.2*cm, subtitle)


def add_two_images_page(c, img_top, img_bottom, page_title, subtitle):
    W, H = landscape(A4)
    c.setPageSize((W, H))
    draw_title(c, page_title, subtitle)

    margin_x = 1.2*cm
    top_y = H - 3.0*cm
    available_h = H - 3.6*cm - 1.0*cm
    img_h = available_h / 2.0
    img_w = W - 2*margin_x

    if os.path.exists(img_top):
        c.drawImage(ImageReader(img_top), margin_x, top_y - img_h,
                    width=img_w, height=img_h, preserveAspectRatio=True, anchor='c')
    if os.path.exists(img_bottom):
        c.drawImage(ImageReader(img_bottom), margin_x, top_y - 2*img_h - 0.4*cm,
                    width=img_w, height=img_h, preserveAspectRatio=True, anchor='c')

    c.showPage()


def add_summary_page(c, summary_kpis, portfolio_compare_all, alerts, week_end):
    W, H = landscape(A4)
    c.setPageSize((W, H))
    draw_title(c, "Weekly Summary",
               f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}  |  Week End: {(week_end.strftime('%Y-%m-%d') if str(week_end)!='NaT' else '')}")

    y = H - 4.0*cm
    c.setFont("Helvetica-Bold", 12)
    c.drawString(1.2*cm, y, "Key Stats")
    y -= 0.6*cm

    c.setFont(CH_FONT, 11)
    for _, r in summary_kpis.iterrows():
        c.drawString(1.2*cm, y, f"{r['項目']}: {r['值']}")
        y -= 0.55*cm

    y -= 0.4*cm
    c.setFont("Helvetica-Bold", 12)
    c.drawString(1.2*cm, y, "Portfolio Comparison")
    y -= 0.6*cm

    c.setFont(CH_FONT, 10)
    c.drawString(1.2*cm, y, "投組 | CAGR | 年化波動 | 最大回撤 | Sharpe")
    y -= 0.5*cm

    for _, row in portfolio_compare_all.iterrows():
        cagr = f"{row.get('CAGR', np.nan):.2%}" if pd.notna(row.get('CAGR', np.nan)) else ""
        vol  = f"{row.get('年化波動', np.nan):.2%}" if pd.notna(row.get('年化波動', np.nan)) else ""
        mdd  = f"{row.get('最大回撤', np.nan):.2%}" if pd.notna(row.get('最大回撤', np.nan)) else ""
        shp  = f"{row.get('Sharpe(簡化)', np.nan):.2f}" if pd.notna(row.get('Sharpe(簡化)', np.nan)) else ""
        line = f"{row['投組']} | {cagr} | {vol} | {mdd} | {shp}"
        c.drawString(1.2*cm, y, line[:180])
        y -= 0.45*cm
        if y < 7.0*cm:
            break

    y -= 0.4*cm
    c.setFont("Helvetica-Bold", 12)
    c.drawString(1.2*cm, y, f"Alerts (Top 10, Total: {0 if alerts is None else len(alerts)})")
    y -= 0.6*cm

    c.setFont(CH_FONT, 10)
    if alerts is None or alerts.empty:
        c.drawString(1.2*cm, y, "無告警 ✅")
    else:
        for _, row in alerts.head(10).iterrows():
            sev = row.get("嚴重度分數", np.nan)
            sev_txt = f"{sev:.2%}" if pd.notna(sev) else ""
            dd_txt = f"{row.get('目前回撤', np.nan):.2%}" if pd.notna(row.get('目前回撤', np.nan)) else ""
            c.drawString(1.2*cm, y, f"- [{row['類別']}] {row['標的']} | Severity {sev_txt} | DD {dd_txt}")
            y -= 0.45*cm
            if y < 2.0*cm:
                break

    c.showPage()


c = canvas.Canvas(OUTPUT_PDF, pagesize=landscape(A4))
add_summary_page(c, summary_kpis, portfolio_compare_all, alerts, week_end)
add_two_images_page(c, png_port, png_alerts, "Charts - Page 1", "Portfolio NAV + Alerts")
add_two_images_page(c, png_scatter, png_corr, "Charts - Page 2", "Risk/Return + Correlation")
c.save()
OUTPUT_PDF


---
## 完成
輸出：
- `weekly_fund_report.xlsx`
- `weekly_fund_report.pdf`（使用思源黑體；若字型檔不存在則自動 fallback）
